# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**One row = one content item, over a trailing 90-day window + comparison 30-day windows.**

- **Grain**: `content_id` (unique content identifier)
- **Historical window**: 90 days of trailing performance (`impressions_90d`, `clicks_90d`, etc.)
- **Recent windows for trend detection**: 
  - Last 30 days (`impressions_last_30d`, `clicks_last_30d`, etc.)
  - Previous 30 days (`impressions_prev_30d`, `clicks_prev_30d`, etc.)
  - These windows allow computation of `trend_direction` and `trend_pct`
- **Content age**: Ranges from 1 day to 502 days old (`content_age_days`)

*Verification below: check grain uniqueness, row count, and content age distribution.*

In [ ]:
import pandas as pd
import numpy as np

# Load the data
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFirst row:\n{df.iloc[0]}")

FileNotFoundError: [Errno 2] No such file or directory: '../data/raw/content_refresh_anonymized.csv'

## 2. Fields: feature / label / context / excluded

### **FEATURES** — knowable BEFORE prediction, safe to use in model
- `search_volume`, `competition`, `competition_level`, `cpc` — keyword metadata
- `content_type`, `main_intent` — content category and intent  
- `word_count`, `char_count` — content length
- `provider_used`, `model_used` — generation provider/model
- `content_age_days`, `age_tier`, `days_since_last_update` — freshness signals
- `freshness_tier`, `word_count_tier`, `char_count_tier` — binned versions

### **LABEL / PROXY** — what we predict or compute it from
- `clicks_90d`, `impressions_90d`, `pageviews_90d`, `sessions_90d`, `users_90d` — direct engagement
- `ctr`, `engagement_rate`, `scroll_rate` — derived engagement rates
- `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d` — recent performance (label windows)

### **CONTEXT** — for grouping/joining/reading, never model input
- `content_id`, `client_id` — pseudonymous identifiers (IDs never are features)
- `age_tier_order`, `impression_tier`, `position_tier` — context tiers for bucketing

### **EXCLUDED** — why they can't be features
- `trend_direction`, `trend_pct`, `is_declining_label` — **LEAKY: derived FROM the label** (computed from impressions_last_30d vs prev_30d)
- `avg_position` — **LEAKY: computed FROM label data** (organic search position is a response to visibility)
- `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d` — **LEAKY: label data** (previous 30 days used to compute trend, confounds recency)
- `engaged_sessions_90d`, `ai_sessions_90d`, `scroll_events_90d`, `days_with_impressions`, `days_with_sessions` — **PARTIALLY OBSERVED**: subset counts from same windows as label (high multicollinearity risk)

In [ ]:
# VERIFY GRAIN: Check that content_id is truly the primary key (no duplicates)
grain_check = df.groupby('content_id').size()
duplicates = grain_check[grain_check > 1]

print("=== GRAIN VERIFICATION ===")
print(f"Total rows: {len(df)}")
print(f"Unique content_ids: {df['content_id'].nunique()}")
print(f"Duplicate content_ids: {len(duplicates)}")
if len(duplicates) == 0:
    print("✓ GRAIN HOLDS: Each content_id appears exactly once")
else:
    print("✗ WARNING: Duplicates detected:", duplicates)

# Content age distribution
print("\n=== CONTENT AGE (time window verification) ===")
print(f"Min content age: {df['content_age_days'].min()} days")
print(f"Max content age: {df['content_age_days'].max()} days")
print(f"Mean content age: {df['content_age_days'].mean():.1f} days")
print(f"Missing content_age: {df['content_age_days'].isna().sum()} rows")

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# VERIFY FIELD CLASSIFICATION: Check missingness patterns and confirm bucket assignments
print("=== MISSINGNESS BY FIELD (%) ===\n")

# Feature fields
feature_fields = ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 
                  'provider_used', 'model_used', 'content_age_days', 'days_since_last_update']
print("FEATURES:")
for col in feature_fields:
    missing_pct = (df[col].isna().sum() / len(df)) * 100
    print(f"  {col}: {missing_pct:.1f}%")

# Label fields
label_fields = ['clicks_90d', 'impressions_90d', 'engagement_rate', 'ctr', 'scroll_rate']
print("\nLABELS:")
for col in label_fields:
    missing_pct = (df[col].isna().sum() / len(df)) * 100
    print(f"  {col}: {missing_pct:.1f}%")

# Excluded (leaky) fields - verify they correlate with label
excluded_fields = ['trend_direction', 'trend_pct', 'avg_position']
print("\nEXCLUDED (LEAKY):")
for col in excluded_fields:
    if col in df.columns:
        missing_pct = (df[col].isna().sum() / len(df)) * 100
        print(f"  {col}: {missing_pct:.1f}%")

# Verify content_type distribution (known source of missingness)
print("\n=== MISSINGNESS BY CONTENT_TYPE ===")
print(df.groupby('content_type')[['word_count', 'search_volume', 'cpc']].apply(
    lambda x: x.isna().sum() / len(x) * 100
).round(1))

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# VERIFY DATA LIMITS: What this dataset CANNOT tell us
print("=== DATA LIMITS — What we can NOT infer ===\n")

# 1. Client imbalance
client_counts = df['client_id'].value_counts()
print("1. CLIENT IMBALANCE:")
print(f"   - Clients in dataset: {df['client_id'].nunique()}")
print(f"   - Median content per client: {client_counts.median():.0f}")
print(f"   - Max content per client: {client_counts.max()}")
print(f"   - Min content per client: {client_counts.min()}")
print("   → Findings: Some clients have >1000 items, some <10.")
print("   → Limit: Client-level causal effects will be confounded by data volume.\n")

# 2. Content type distribution
content_type_dist = df['content_type'].value_counts()
print("2. CONTENT TYPE DISTRIBUTION:")
print(df['content_type'].value_counts())
print(f"   → Limit: ~90% are 'keyword article'; feedly/comparison are rare (~1% each).\n")

# 3. Sparse performance
zero_impression_rows = (df['impressions_90d'] == 0).sum()
zero_click_rows = (df['clicks_90d'] == 0).sum()
print("3. SPARSE PERFORMANCE:")
print(f"   - Rows with 0 impressions (90d): {zero_impression_rows} ({100*zero_impression_rows/len(df):.1f}%)")
print(f"   - Rows with 0 clicks (90d): {zero_click_rows} ({100*zero_click_rows/len(df):.1f}%)")
print("   → Limit: Cannot measure engagement for ~30% of content; these may simply be new/hidden.\n")

# 4. Window overlap
print("4. WINDOW ALIGNMENT:")
print(f"   - Last 30d + Prev 30d do NOT align with 90d window (different aggregation)")
print(f"   - Trend computed from last_30d vs prev_30d only — not anchored to calendar dates")
print("   → Limit: Cannot join with external calendar events without explicit date columns.\n")

# 5. Age distribution
print("5. CONTENT AGE:")
print(f"   - Mean age: {df['content_age_days'].mean():.1f} days")
print(f"   - Median age: {df['content_age_days'].median():.1f} days")
print(f"   - % <30 days old: {100*(df['content_age_days'] < 30).sum()/len(df):.1f}%")
print("   → Limit: Mostly mature content; can't study 'first 7 days' dynamics.")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.